# 👟 Google BlazeFoot 4-KP Training Pipeline on Google Colab
This notebook trains the **4-Keypoint Native Google BlazeFoot (MediaPipe style)** detector on Google Colab (Tesla T4 / A100 GPU) with full **Google Drive synchronization**.

### Complete Workflow:
1. **Mount Google Drive** (to save datasets, checkpoints, and ONNX models permanently).
2. **Clone GitHub Repository & Install Requirements**.
3. **Download Dataset from Roboflow + Auto Repair & Shuffle** (or load from Drive cache).
4. **Train Google BlazeFoot (100 Epochs)** on GPU.
5. **Export to ONNX** (FP32, FP16, INT8) & Auto-Sync deliverables to Drive.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create permanent experiment directory in your Google Drive
DRIVE_DIR = '/content/drive/MyDrive/Shoes_VTO_Experiments'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"✅ Google Drive Mounted! Permanent storage at: {DRIVE_DIR}")

### Step 2: Clone GitHub Repository & Install Dependencies

In [ ]:
%cd /content
!rm -rf Shoes_VTO
!git clone https://github.com/HagAli22/Shoes_VTO.git
%cd Shoes_VTO

# Install dependencies
!pip install -q albumentations onnx onnxruntime onnxconverter-common roboflow opencv-python-headless

import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")

### Step 3: Download from Roboflow, Remap, Repair & Shuffle Dataset
*(If already cached in Google Drive, it will load instantly)*

In [ ]:
import shutil
from pathlib import Path

drive_data_zip = f'{DRIVE_DIR}/shuffled_v3.zip'

if os.path.exists(drive_data_zip):
    print("📦 Found cached dataset in Google Drive! Extracting...")
    !unzip -q "{drive_data_zip}" -d data/
    print("✅ Dataset restored from Drive cache!")
else:
    print("⬇️ Downloading dataset from Roboflow...")
    # Replace with your Roboflow API key if needed
    ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
    
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace("shoes-vto").project("fingers_keypoint")
        version = project.version(12)
        dataset = version.download("yolov8", location="data/shoes_v3_raw")
    except Exception as e:
        print(f"Note on Roboflow API: {e}")
        print("Using GitHub repository's included data or manual download...")

    # Remap classes (0->1, 1->0 for left/right convention), repair coordinates, and create stratified shuffle
    print("🔧 Running clean coordinate repair and 80/10/10 stratified shuffle...")
    !python tools/prepare_shuffled_dataset.py

    # Cache to Google Drive for future instant runs
    print("💾 Caching clean dataset to Google Drive...")
    !zip -q -r "{drive_data_zip}" data/shuffled_v3
    print(f"✅ Dataset cached to {drive_data_zip}")

# Verify dataset count
print("\n--- Dataset Summary ---")
!python -c "import os; print('Train images:', len(os.listdir('data/shuffled_v3/train/images'))); print('Val images  :', len(os.listdir('data/shuffled_v3/valid/images'))); print('Test images :', len(os.listdir('data/shuffled_v3/test/images')))"

### Step 4: Train Google BlazeFoot (4-KP Native Architecture) on GPU

In [ ]:
!python -m src.models.blazefoot.train_blazefoot \
  --data "data/shuffled_v3" \
  --epochs 100 \
  --batch 32 \
  --lr0 0.001 \
  --device 0 \
  --project "outputs/stage_a" \
  --name "blazefoot_4kp_colab"

# Save best PyTorch checkpoint to Google Drive
!cp outputs/stage_a/blazefoot_4kp_colab/weights/best.pt "{DRIVE_DIR}/checkpoints/blazefoot_4kp_best.pt"
print(f"\n✅ Best checkpoint saved to Google Drive: {DRIVE_DIR}/checkpoints/blazefoot_4kp_best.pt")

### Step 5: Export to ONNX (FP32, FP16, INT8) & Save to Drive

In [ ]:
!python -m src.export.export_blazefoot \
  --weights "outputs/stage_a/blazefoot_4kp_colab/weights/best.pt" \
  --output_dir "shoes-vto-ai-v2/models" \
  --prefix "stage-a-320-blazefoot"

# Copy all exported models to Google Drive
!cp shoes-vto-ai-v2/models/stage-a-320-blazefoot* "{DRIVE_DIR}/models/"
print(f"\n✅ Exported ONNX models synchronized to Google Drive: {DRIVE_DIR}/models/")

# List models with sizes in Drive
!ls -lh "{DRIVE_DIR}/models/"

### Step 6: Download Artifacts to Local PC (Optional)

In [ ]:
from google.colab import files

# Download the lightweight FP16 (1.19 MB) and INT8 (0.79 MB) models directly
files.download('shoes-vto-ai-v2/models/stage-a-320-blazefoot-fp16.onnx')
files.download('shoes-vto-ai-v2/models/stage-a-320-blazefoot-int8.onnx')
files.download('outputs/stage_a/blazefoot_4kp_colab/weights/best.pt')